# core

> Tells, findings, the rule registry, and what the meter cannot see

In [1]:
#| default_exp core

In [2]:
#| hide
from nbdev.showdoc import *

slopometer measures prose against the reference-prose rules in `aai_coding.write_docs`: 26 numbered tells, the named ways reference prose goes wrong. This module defines what every rule shares: the table of tells, the `Finding` record every detector returns, the registry that rules join at import time, and the registry of tells no deterministic rule can detect.

In [3]:
#| export
from fastcore.utils import *

In [4]:
from fastcore.test import *

## The tells

In [5]:
#| export
tells = {
    1: 'splices',              2: 'contract by aside',       3: 'emphasis devices',
    4: 'elegant variation',    5: 'flourish over identifier', 6: 'consequence glue',
    7: 'hedging',              8: 'noting fillers',          9: 'restatement',
    10: 'justification rider', 11: 'decorative verbs',       12: 'audience misjudged',
    13: 'throat-clearing',     14: "today's-world opener",   15: 'announce-then-deliver',
    16: 'not-X-but-Y',         17: 'teaser pivot',           18: 'rhetorical questions',
    19: 'filler transitions',  20: 'forced symmetry',        21: 'appraisal preamble',
    22: 'artifact-as-agent',   23: 'recipient-as-subject',   24: 'decoration',
    25: 'over-structuring',    26: 'false depth'}

The numbers are `write_docs`'s own, and its docstring stays the source for what each tell means. They are append-only, so a rule bound to a number stays correct across revisions. slopometer names the tells and never restates them. Rules built on `write_docs` guidance that sits outside the numbered tells, like the banned-word lists and the active-voice preference, carry `tell=None` and display without a tell clause.

## Findings

In [6]:
#| export
KILL,SMELL,PRESSURE = 10,3,1

The tiers span an order of magnitude on purpose. One kill-on-sight finding outweighs any accumulation of pressure findings. Calibration against the `write_docs` passage pair will set the exact numbers. The tiers are the commitment.


In [7]:
#| export
class Finding:
    def __init__(self,
        rule, # Name of the rule that fired
        tell, # `write_docs` tell number the rule detects
        start, # Character offset where the span starts, within the text the detector received
        end, # Character offset one past the span's end
        text, # The offending span itself
        weight, # The rule's tier, escalated where the rule escalates
        suggestion=None, # A mechanical replacement, when one exists
    ): store_attr()
    def __repr__(self):
        sug = f' -> {self.suggestion!r}' if self.suggestion else ''
        tl = '' if self.tell is None else f' (tell {self.tell}, {tells[self.tell]})'
        return f'[{self.weight}] {self.rule}{tl}: {self.text!r}{sug}'

A `Finding` is one rule firing on one span. `start` and `end` are offsets into the text the detector received. The runner rebases them to file positions when it assembles a report. The span text is stored too, and a finding therefore displays without its source. Here is a finding built by hand, for the sentence the meter exists to catch:

In [8]:
s = "It isn't just a poller - it's the liveness authority."
Finding('notxbuty', 16, 0, len(s), s, KILL)

[10] notxbuty (tell 16, not-X-but-Y): "It isn't just a poller - it's the liveness authority."

## The rule registry

In [9]:
#| export
rules = {}

def rule(
    name, # Rule name, unique within slopometer
    tell, # `write_docs` tell number the rule detects
    weight, # Tier the rule's findings carry
    level, # The unit the detector reads: 'phrase', 'sentence', 'para', or 'doc'
):
    "Register a detector, a function from its unit's text to the list of `Finding`s in it"
    def _f(f):
        f.tell,f.weight,f.level = tell,weight,level
        rules[name] = f
        return f
    return _f

Rules join `rules` at import time through the decorator, which stamps the metadata that the runner and the drift test read. A detector receives one unit of text at its level and nothing wider. That locality is a design rule from the PRD: the paragraph is the largest unit of linguistic analysis.

The simplest rule in the package needs no parse and no lexicon, and it serves here as the registry's working example: tell 24, decoration. Reference prose is ASCII. Emoji and ornamental symbols are always wrong in it, which makes this a kill-on-sight rule, and arrow glyphs get a mechanical suggestion. Accented letters are ordinary spelling, not decoration. The em dash is a splice (tell 1), not decoration, and a different rule owns it. Both pass here, and the first line of the example is the precision commitment: clean text must produce no findings.

In [10]:
#| export
_ornament = re.compile(r'[\u2190-\u2bff\ufe0f\U0001f000-\U0001faff]+')
_arrows = {'→':'->', '⇒':'=>', '←':'<-', '↔':'<->'}

@rule('decoration', tell=24, weight=KILL, level='phrase')
def find_decoration(txt):
    "Flag emoji and ornamental unicode, suggesting ASCII for arrows"
    return [Finding('decoration', 24, m.start(), m.end(), m.group(), KILL, _arrows.get(m.group()))
        for m in _ornament.finditer(txt)]

In [11]:
test_eq(find_decoration('naïve café — no decoration, only accents'), [])
find_decoration('Done ✅ ship it 🚀🎉 then → prod')

[[10] decoration (tell 24, decoration): '✅',
 [10] decoration (tell 24, decoration): '🚀🎉',
 [10] decoration (tell 24, decoration): '→' -> '->']

## What the meter cannot see

Seven entries record judgment no deterministic rule reaches. Recording them with reasons serves two consumers. The drift test in the scoring notebook asserts that every `write_docs` tell number maps to a rule or to an entry here, and a tell added to `write_docs` without a slopometer decision fails it. Readers of `check_docs` reviews learn what the meter cannot see. Tells 9, 11, and 12 appear in both places on purpose: a rule covers their definable half, and this registry records the rest.

In [12]:
#| export
unscoreable = {
    2: 'knowing which fact is the contract takes judgment',
    5: "telling a flourish from a real identifier takes the project's vocabulary",
    9: 'lead-sentence restatement defeats averaged sentence vectors, and only the heading echo is detectable (measured in the para notebook)',
    10: 'a justification rider reads like a stated consequence, and only the domain says which it is',
    11: 'verb texture beyond the banned-verb lexicon is a whiteboard judgment',
    12: 'matching prose to its audience takes the audience, and the coinage rule covers only the definable half',
    26: 'depth is judged against what the reader already knows',}

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()